In [1]:
!pip install bitsandbytes>=0.43.0 transformers>=4.38.0 peft>=0.7.0 accelerate>=0.25.0 trl>=0.7.4 datasets>=2.14.0

In [2]:
import sys
import os
import logging
import torch
import torch.distributed as dist
import importlib.util
import subprocess
from accelerate import notebook_launcher
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import get_peft_model, LoraConfig, TaskType
from datasets import load_dataset
from trl import SFTTrainer, SFTConfig
from torch.distributed.fsdp import (
    FullyShardedDataParallel as FSDP,
    MixedPrecision,
    CPUOffload,
    BackwardPrefetch,
    ShardingStrategy,
)
import transformers
from transformers.models.llama.modeling_llama import LlamaDecoderLayer, LlamaRMSNorm, LlamaRotaryEmbedding
import math
import torch.nn.init as init
import torch.nn as nn
from functools import partial
from torch.distributed.fsdp.wrap import size_based_auto_wrap_policy

# Set up logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(name)s - %(levelname)s - %(message)s',
    handlers=[logging.StreamHandler(sys.stdout)]
)
logger = logging.getLogger(__name__)

# Check if a package is installed
def is_package_installed(package_name):
    """Check if a package is installed"""
    return importlib.util.find_spec(package_name) is not None

# Install missing packages
def install_missing_packages():
    """Install required packages if missing"""
    required_packages = [
        "bitsandbytes>=0.43.0",
        "transformers>=4.38.0",
        "peft>=0.7.0",
        "accelerate>=0.25.0",
        "trl>=0.7.4",
        "datasets>=2.14.0",
    ]
    
    for package in required_packages:
        package_name = package.split(">=")[0]
        if not is_package_installed(package_name):
            logger.info(f"Installing {package}...")
            try:
                subprocess.check_call([sys.executable, "-m", "pip", "install", package])
                logger.info(f"Successfully installed {package}")
            except subprocess.CalledProcessError as e:
                logger.error(f"Failed to install {package}: {e}")
                if package_name == "bitsandbytes":
                    logger.info("Attempting to install bitsandbytes with specific CUDA version...")
                    try:
                        subprocess.check_call([sys.executable, "-m", "pip", "install", "bitsandbytes-cuda118"])
                    except:
                        logger.error("Failed to install bitsandbytes-cuda118. Please install manually.")

# Set up environment variables
def setup_environment():
    """Set up environment variables for training"""
    os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"
    os.environ["CUDA_VISIBLE_DEVICES"] = "0,1"
    os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True,roundup_power2_divisions:[32:256,64:128,256:64,>:32]"
    os.environ["ACCELERATE_USE_FSDP"] = "1"
    os.environ["FSDP_CPU_RAM_EFFICIENT_LOADING"] = "1"
    os.environ["PYTORCH_NO_CUDA_MEMORY_CACHING"] = "1"
    logger.info("Environment variables set")

# Main training function for distributed execution
def training_function():
    """Main training function to be executed by notebook_launcher"""
    # Get local rank for distributed training
    local_rank = int(os.environ.get("LOCAL_RANK", "0"))
    world_size = int(os.environ.get("WORLD_SIZE", "1"))
    
    logger.info(f"Starting process with local_rank: {local_rank}, world_size: {world_size}")
    
    # Set device
    torch.cuda.set_device(local_rank)
    
    # IMPORTANT: Initialize the process group for distributed training
    # This is required before using FSDP
    if not dist.is_initialized():
        logger.info("Initializing process group for distributed training...")
        # If using notebook_launcher, it should set the required env vars
        dist.init_process_group(backend="nccl")
        logger.info(f"Process group initialized: rank={dist.get_rank()}, world_size={dist.get_world_size()}")
    
    # Log system information
    logger.info(f"PyTorch version: {torch.__version__}")
    logger.info(f"CUDA available: {torch.cuda.is_available()}")
    if torch.cuda.is_available():
        logger.info(f"CUDA device count: {torch.cuda.device_count()}")
        logger.info(f"Current CUDA device: {torch.cuda.current_device()}")
        logger.info(f"Current CUDA device name: {torch.cuda.get_device_name(torch.cuda.current_device())}")
    
    # Verify bitsandbytes installation
    try:
        import bitsandbytes as bnb
        logger.info(f"Successfully imported bitsandbytes version: {bnb.__version__}")
    except ImportError:
        logger.error("Failed to import bitsandbytes. Attempting reinstallation...")
        try:
            subprocess.check_call([sys.executable, "-m", "pip", "install", "--force-reinstall", "bitsandbytes>=0.43.0"])
            import bitsandbytes as bnb
            logger.info(f"Reinstalled and imported bitsandbytes version: {bnb.__version__}")
        except:
            logger.error("Failed to reinstall bitsandbytes. Training may fail.")
    
    # Setup model and tokenizer
    logger.info("Setting up model and tokenizer...")
    dtype = torch.float16
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_use_double_quant=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=dtype,
        bnb_4bit_quant_storage=dtype,  # For FSDP compatibility
    )
    
    # For T4 GPUs, we need to be careful with memory usage
    logger.info("Loading model...")
    model_name = "unsloth/meta-Llama-3.1-8B-Instruct-bnb-4bit"
    try:
        model = AutoModelForCausalLM.from_pretrained(
            model_name,
            quantization_config=bnb_config,
            torch_dtype=dtype,
            device_map=None,  # Important for FSDP
            attn_implementation="sdpa",
            low_cpu_mem_usage=True,
        )
    except Exception as e:
        logger.error(f"Error loading model: {e}")
        try:
            logger.info("Attempting to load with alternate configuration...")
            model = AutoModelForCausalLM.from_pretrained(
                model_name,
                torch_dtype=dtype,
                device_map=None,
                attn_implementation="sdpa",
                low_cpu_mem_usage=True,
            )
        except Exception as e2:
            logger.error(f"Failed with alternate config too: {e2}")
            return
    
    logger.info("Loading tokenizer...")
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    tokenizer.padding_side = "right"
    
    # Define mixed precision policy - must match quantization storage type
    mixed_precision_policy = MixedPrecision(
        param_dtype=dtype,
        reduce_dtype=dtype,
        buffer_dtype=dtype,
    )
    
    # CPU offloading for memory efficiency on limited GPUs
    cpu_offload = CPUOffload(offload_params=True)
    
    # Set up LoRA
    logger.info("Setting up LoRA configuration...")
    lora_config = LoraConfig(
        r=32,  # Reduced from 64 to save memory on T4 GPUs
        lora_alpha=64,  # Reduced from 128 for better T4 compatibility
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj", 
                        "gate_proj", "up_proj", "down_proj"],
        lora_dropout=0,
        bias="none",
        task_type=TaskType.CAUSAL_LM,
    )
    
    # Apply LoRA
    logger.info("Applying LoRA to model...")
    model = get_peft_model(model, lora_config)
    
    # Set only LoRA parameters to trainable
    logger.info("Setting trainable parameters...")
    trainable_param_count = 0
    total_param_count = 0
    with torch.no_grad():
        for name, param in model.named_parameters():
            total_param_count += param.numel()
            if ".lora_A." in name or ".lora_B." in name:
                param.requires_grad_(True)
                trainable_param_count += param.numel()
            else:
                param.requires_grad_(False)
    logger.info(f"Trainable parameters: {trainable_param_count:,} ({trainable_param_count/total_param_count:.2%} of total {total_param_count:,})")
    
    # Enable gradient checkpointing for memory efficiency
    logger.info("Enabling gradient checkpointing...")
    model.gradient_checkpointing_enable()
    model.enable_input_require_grads()
    
    # Patch LlamaRMSNorm to add reset_parameters method
    logger.info("Patching LlamaRMSNorm to add reset_parameters method...")
    
    # Add reset_parameters method to LlamaRMSNorm if it doesn't exist already
    if not hasattr(LlamaRMSNorm, 'reset_parameters'):
        def reset_parameters(self):
            # RMSNorm doesn't need parameter initialization beyond what's done in __init__
            pass
        
        # Add the method to the class
        LlamaRMSNorm.reset_parameters = reset_parameters
        logger.info("Successfully added reset_parameters method to LlamaRMSNorm")
    
    # Patch LlamaRotaryEmbedding to add reset_parameters method
    logger.info("Patching LlamaRotaryEmbedding to add reset_parameters method...")
    
    # Add reset_parameters method to LlamaRotaryEmbedding if it doesn't exist already
    if not hasattr(LlamaRotaryEmbedding, 'reset_parameters'):
        def reset_parameters(self):
            # RotaryEmbedding doesn't need parameter initialization beyond what's done in __init__
            pass
        
        # Add the method to the class
        LlamaRotaryEmbedding.reset_parameters = reset_parameters
        logger.info("Successfully added reset_parameters method to LlamaRotaryEmbedding")
    
    # Patch Linear layers to safely handle quantized weights
    logger.info("Patching modules to safely handle quantized weights...")
    
    # Create a patch for any module with weight tensors
    def patch_module_with_quantized_safety(module_class):
        if hasattr(module_class, 'reset_parameters'):
            # Store the original reset_parameters method
            original_reset_parameters = module_class.reset_parameters
            
            def safe_reset_parameters(self):
                """A safe reset_parameters that checks for quantized weights first"""
                skip_init = False
                
                # Check if weights are quantized (byte, int8, etc.)
                if hasattr(self, 'weight') and self.weight is not None:
                    if self.weight.dtype in [torch.uint8, torch.int8, torch.quint8, torch.qint8]:
                        logger.info(f"Skipping initialization for quantized weights in {self}")
                        skip_init = True
                
                if not skip_init:
                    # Call the original initialization for non-quantized weights
                    original_reset_parameters(self)
            
            # Replace the method
            module_class.reset_parameters = safe_reset_parameters
            return True
        else:
            # For modules without reset_parameters, add an empty one
            def empty_reset_parameters(self):
                pass
            
            module_class.reset_parameters = empty_reset_parameters
            return False
    
    # Patch common module types
    for module_class in [nn.Linear, nn.Conv1d, nn.Conv2d, nn.Embedding]:
        had_reset = patch_module_with_quantized_safety(module_class)
        if had_reset:
            logger.info(f"Patched {module_class.__name__}.reset_parameters to handle quantized weights")
        else:
            logger.info(f"Added empty reset_parameters to {module_class.__name__}")
    
    # Convert frozen quantized parameters to buffers to avoid dtype mismatch during FSDP flattening
    logger.info("Converting frozen quantized parameters to buffers...")
    for module in model.modules():
        for name, param in list(module.named_parameters(recurse=False)):
            if not param.requires_grad and param.dtype in [torch.uint8, torch.int8, torch.quint8, torch.qint8]:
                logger.info(f"Converting {name} in {module} from parameter to buffer")
                module._parameters.pop(name)
                module.register_buffer(name, param)
    
    # Mark quantized parameters to be ignored by FSDP (if any remain as parameters)
    for name, param in model.named_parameters():
        if param.dtype in [torch.uint8, torch.int8, torch.quint8, torch.qint8]:
            setattr(param, '_fsdp_ignore', True)
    
    # Cast the entire model to torch.float16 to ensure uniform dtypes for FSDP flattening
    logger.info("Casting model to torch.float16 for uniform dtypes...")
    model = model.half()
    
    # Reduce dataset size and sequence length further for T4 GPUs
    max_seq_length = 512  # Reduced for faster training
    logger.info(f"Using max sequence length: {max_seq_length}")
    
    # Load and prepare dataset
    logger.info("Loading dataset...")
    try:
        url = "https://huggingface.co/datasets/laion/OIG/resolve/main/unified_chip2.jsonl"
        dataset = load_dataset("json", data_files={"train": url}, split="train[:2%]")
        
        # Ensure text field is properly formatted
        def format_text(example):
            text = example["text"]
            # Handle nested lists and ensure string format
            if isinstance(text, list):
                return {"text": " ".join([str(t) for t in text])}
            return {"text": str(text)}
        
        dataset = dataset.map(format_text, remove_columns=[col for col in dataset.column_names if col != "text"])
        
        # Tokenize dataset upfront with proper padding/truncation
        def tokenize_function(examples):
            return tokenizer(
                examples["text"],
                padding="max_length",
                truncation=True,
                max_length=max_seq_length,
                return_tensors="pt",
            )
        
        dataset = dataset.map(
            tokenize_function,
            batched=True,
            remove_columns=["text"],
            desc="Tokenizing dataset",
        )

    except Exception as e:
        logger.error(f"Error loading dataset: {e}")
        import traceback
        logger.error(traceback.format_exc())
        return
    
    # Configure training with FSDP2 settings
    logger.info("Setting up FSDP2 configuration...")
    fsdp_config = {
        "fsdp_transformer_layer_cls_to_wrap": ["LlamaDecoderLayer"],
        "fsdp_sharding_strategy": 1,  # SHARD_GRAD_OP
        "fsdp_offload_params": True,
        "fsdp_activation_checkpointing": True,
        "fsdp_use_orig_params": True,
        "fsdp_sync_module_states": True,
        "fsdp_cpu_ram_efficient_loading": True,
    }

    training_args = SFTConfig(
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        warmup_steps=1,
        max_steps=20,
        logging_steps=1,
        output_dir="outputs",
        seed=3407,
        max_seq_length=max_seq_length,
        fp16=dtype == torch.float16,
        bf16=dtype == torch.bfloat16,
        report_to="none",
        save_steps=10,
        save_total_limit=2,
        # FSDP2 configuration
        fsdp="full_shard auto_wrap offload",
        fsdp_config=fsdp_config,
        # Critical data processing settings
        remove_unused_columns=False,  # Prevent column removal
        dataloader_num_workers=0,
        disable_tqdm=False,
        logging_first_step=True,
        dataset_num_proc=1,
        dataloader_drop_last=False,
        dataloader_pin_memory=False,
        auto_find_batch_size=True,
    )

    # Initialize trainer with proper data handling
    logger.info("Initializing trainer...")
    trainer = SFTTrainer(
        model=model,
        train_dataset=dataset,
        args=training_args,
        tokenizer=tokenizer,
        data_collator=transformers.DataCollatorForLanguageModeling(
            tokenizer=tokenizer,
            mlm=False,
            pad_to_multiple_of=8
        ),
        # Version-safe text processing
        formatting_func=lambda example: [example["text"]],
    )

    # Run training
    logger.info("Starting training...")
    try:
        trainer.train()
        # Save the model (LoRA adapters)
        if local_rank == 0:
            logger.info("Training complete! Saving model...")
            trainer.save_model()
            logger.info("Model saved successfully.")
    except Exception as e:
        logger.error(f"Error during training: {e}")
        import traceback
        traceback.print_exc()

# Main execution script for Kaggle notebook
def main():
    # Install required packages
    install_missing_packages()
    
    # Set up environment variables for T4 GPUs
    setup_environment()
    
    # Make sure accelerate is imported
    try:
        from accelerate import notebook_launcher
    except ImportError:
        logger.error("accelerate library is missing. Installing...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "accelerate>=0.25.0"])
        from accelerate import notebook_launcher
    
    # Set required environment variables for PyTorch Distributed
    os.environ["MASTER_ADDR"] = "localhost"
    os.environ["MASTER_PORT"] = "29500"
    
    # Launch training with notebook_launcher
    logger.info("Launching training across 2 GPUs...")
    notebook_launcher(training_function, num_processes=2)
    logger.info("Training complete!")

# Call main() to execute
main()

Launching training on 2 GPUs.


Unused kwargs: ['_load_in_4bit', '_load_in_8bit', 'quant_method']. These kwargs are not used in <class 'transformers.utils.quantization_config.BitsAndBytesConfig'>.
Unused kwargs: ['_load_in_4bit', '_load_in_8bit', 'quant_method']. These kwargs are not used in <class 'transformers.utils.quantization_config.BitsAndBytesConfig'>.
/usr/local/lib/python3.10/dist-packages/transformers/quantizers/auto.py:186: UserWarning: You passed `quantization_config` or equivalent parameters to `from_pretrained` but the model you're loading already has a `quantization_config` attribute. The `quantization_config` from the model will be used.
  warnings.warn(warning_msg)
/usr/local/lib/python3.10/dist-packages/transformers/quantizers/auto.py:186: UserWarning: You passed `quantization_config` or equivalent parameters to `from_pretrained` but the model you're loading already has a `quantization_config` attribute. The `quantization_config` from the model will be used.
  warnings.warn(warning_msg)


model.safetensors:   0%|          | 0.00/5.70G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/55.5k [00:00<?, ?B/s]

...

/usr/local/lib/python3.10/dist-packages/trl/trainer/sft_trainer.py:381: UserWarning: You passed a dataset that is already processed (contains an `input_ids` field) together with a formatting function. Therefore `formatting_func` will be ignored. Either remove the `formatting_func` or pass a dataset that is not already processed.
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/accelerate/accelerator.py:1585: UserWarning: Upcasted low precision parameters in Linear because mixed precision turned on in FSDP. Affects: weight.
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/accelerate/accelerator.py:1591: UserWarning: FSDP upcast of low precision parameters may affect the precision of model checkpoints.
  warnings.warn(
`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.
`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.


Step,Training Loss
1,8.506900
2,9.983400
3,8.818500
4,7.822600
5,10.544200
6,8.186700
7,7.737500
8,9.111200
9,8.172100


Step,Training Loss
1,8.506900
2,9.983400
3,8.818500
4,7.822600
5,10.544200
6,8.186700
7,7.737500
8,9.111200
9,8.172100


/usr/local/lib/python3.10/dist-packages/torch/distributed/fsdp/fully_sharded_data_parallel.py:690: FutureWarning: FSDP.state_dict_type() and FSDP.set_state_dict_type() are being deprecated. Please use APIs, get_state_dict() and set_state_dict(), which can support different parallelisms, FSDP1, FSDP2, DDP. API doc: https://pytorch.org/docs/stable/distributed.checkpoint.html#torch.distributed.checkpoint.state_dict.get_state_dict .Tutorial: https://pytorch.org/tutorials/recipes/distributed_checkpoint_recipe.html .
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/torch/distributed/fsdp/fully_sharded_data_parallel.py:690: FutureWarning: FSDP.state_dict_type() and FSDP.set_state_dict_type() are being deprecated. Please use APIs, get_state_dict() and set_state_dict(), which can support different parallelisms, FSDP1, FSDP2, DDP. API doc: https://pytorch.org/docs/stable/distributed.checkpoint.html#torch.distributed.checkpoint.state_dict.get_state_dict .Tutorial: https://pytorch.org/tuto

ChildFailedError: 
===================================================
training_function FAILED
---------------------------------------------------
Failures:
  <NO_OTHER_FAILURES>
---------------------------------------------------
Root Cause (first observed failure):
[0]:
  time      : 2025-03-03_17:50:00
  host      : d731f537bfb0
  rank      : 0 (local_rank: 0)
  exitcode  : -6 (pid: 89)
  error_file: <N/A>
  traceback : Signal 6 (SIGABRT) received by PID 89
===================================================